# CS781 Phase 6 

In [1]:
%%writefile /kaggle/working/phase6_run.py
# %% [markdown]
# # CS781 Capstone — Phase 6: Abstractive Legal Summarization (decoder-only)
# **Base model:** your Phase-1 decoder (RoPE + multi-query attention + RMSNorm + SwiGLU, ~107M, 12k byte-level BPE).
#
# One notebook, six methods. Change **one line** (`METHOD`) in the config cell and run top-to-bottom, once per method:
#
# | `METHOD` | Technique | What is trained |
# |---|---|---|
# | `lorma` | LoRMA (multiplicative low-rank, permutation rank-inflation) — **compulsory** | `B`,`A` per attention projection |
# | `lora` | LoRA (additive low-rank) | `B`,`A` per attention + FFN projection |
# | `houlsby` | Houlsby bottleneck adapters (after attention and after FFN) + norms | adapters + RMSNorm weights |
# | `prefix` | Prefix-tuning (per-layer key/value prefixes, MLP re-parametrisation) | prefix encoder |
# | `prompt` | Prompt-tuning (soft prompt embeddings in front of the input) | soft prompt |
# | `kd` | Knowledge distillation: 12-layer teacher -> 6-layer student initialised from alternate Phase-1 layers | whole student |
#
# **Run order:** `lorma, lora, houlsby, prefix, prompt` first (any order), then `kd` (it needs one finished teacher run — default `lorma`; set `TEACHER` to whichever scored best).
# Every run starts from the Phase-1 weights. Nothing else pretrained is used.
#
# **Attach as Kaggle inputs:** (1) your final Phase-1 output (`model.pt` + `bpe_tokenizer.json`), (2) for `kd`, the output of the teacher run. GPU + Internet ON, `HuggingFace` secret enabled.
#
# **Known limits (read before trusting numbers):** the Phase-1 decoder was trained at 512 tokens. This notebook stretches it to `MAX_CTX` with RoPE position-interpolation; PEFT methods adapt to that only partially. Prompt-tuning on a ~100M model is expected to be the weakest method — that is a finding for the report, not a bug.

# %%
import os, sys, math, json, time, random, zipfile, csv, glob, re, gc, subprocess, copy
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score", "rouge-score", "--break-system-packages"], check=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"
print(DEVICE, torch.cuda.device_count())

# %% [markdown]
# ## 1. Configuration

# %%
METHOD = os.environ.get("P6_METHOD", "lorma")   # "lorma" | "lora" | "houlsby" | "prefix" | "prompt" | "kd"
TEACHER = "lorma"         # only used when METHOD == "kd": which finished run is the teacher (not "prompt")

SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

INPUT_ROOT = os.environ.get("CS781_INPUT", "/kaggle/input")
OUT_ROOT = os.environ.get("CS781_OUT", "/kaggle/working/cs781_phase6")
OUT_DIR = os.path.join(OUT_ROOT, METHOD)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_ROOT, "phase6_results.csv")

# ---- sequence budget (tokens). Phase-1 BPE costs ~2.7 tokens/word because every word is followed by </s>. ----
MAX_CTX = 1024            # document + summary tokens per training example (Phase-1 model was trained at 512)
SUM_TOK = 384             # max summary tokens (incl. end marker). Check the length stats printed in section 3 and adjust!
ROPE_MODE = "interp"      # "interp": linear position interpolation 512 -> MAX_CTX ; "extrap": plain RoPE extrapolation
CONTEXT_MODE = "head_tail"  # long-document handling: "head" | "head_tail" | "salient"
HEAD_FRAC = 0.7           # head_tail: share of the document budget taken from the start (rest from the end)
SEP_WORD, END_WORD = "SUMMARY:", "ENDSUM"   # ordinary words -> work with the frozen Phase-1 embeddings

# ---- optimisation ----
EPOCHS = int(os.environ.get("P6_EPOCHS", 3))
MICRO_BATCH, GRAD_ACCUM = 4, 4            # effective batch 16 ; reduce MICRO_BATCH to 2 if CUDA OOM
WARMUP_RATIO = 0.05
EVAL_EVERY = 100                          # optimiser steps
N_VAL_DOCS = 150                          # held out from train.json (no separate dev set is released)
N_VAL_GEN = 64                            # val docs used for ROUGE/BERTScore after training
TIME_BUDGET_H = float(os.environ.get("P6_TIME_H", 5.0))                       # stop training (and still write a submission) after this many hours
GEN_BATCH = 24

# ---- decoding (a tiny grid is searched on the val docs) ----
DECODE_GRID = [
    dict(temperature=0.0, top_k=0, top_p=1.0, rep_pen=1.0),
    dict(temperature=0.0, top_k=0, top_p=1.0, rep_pen=1.15),
    dict(temperature=0.5, top_k=50, top_p=0.9, rep_pen=1.15),
]

# ---- per-method hyper-parameters ----
CFG = {
    "lorma":   dict(lr=2e-4, rank=16, targets=("q_proj", "k_proj", "v_proj", "out_proj")),
    "lora":    dict(lr=5e-4, rank=16, alpha=32, dropout=0.05,
                    targets=("q_proj", "k_proj", "v_proj", "out_proj", "gate", "up", "down")),
    "houlsby": dict(lr=5e-4, bottleneck=64, train_norms=True),
    "prefix":  dict(lr=2e-3, prefix_len=48, hidden=512),
    "prompt":  dict(lr=5e-3, prompt_len=64),
    "kd":      dict(lr=2e-4, student_layers=(1, 3, 5, 7, 9, 11), temperature=2.0, alpha=0.5),
}
assert METHOD in CFG, f"unknown METHOD {METHOD}"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("run:", METHOD, CFG[METHOD])

# %% [markdown]
# ## 2. Phase-1 assets (model + BPE tokenizer)

# %%
def find_files(name, roots=None):
    roots = roots or [INPUT_ROOT]
    hits = []
    for root in roots:
        for dp, _, fns in os.walk(root, followlinks=True):
            if name in fns: hits.append(os.path.join(dp, name))
    return sorted(hits)

P1_MODEL_OVERRIDE = None   # e.g. "/kaggle/input/<ds>/cs781_phase1/<run>/model.pt"
P1_TOKENIZER_OVERRIDE = None

_cands = [P1_MODEL_OVERRIDE] if P1_MODEL_OVERRIDE else find_files("model.pt")
_cands = [p for p in _cands if "phase6" not in p]
print("Phase-1 model candidates:", _cands)
assert _cands, "No Phase-1 model.pt found: attach your Phase-1 output as an input."
P1, P1_PATH = None, None
for p in _cands:                                    # pick the checkpoint with the most training steps
    ck = torch.load(p, map_location="cpu", weights_only=False)
    if P1 is None or ck["config"].get("step", 0) > P1["config"].get("step", 0): P1, P1_PATH = ck, p
    else: del ck
print("using", P1_PATH, "| Phase-1 steps:", P1["config"].get("step"), "|", {k: v for k, v in P1["config"].items() if k != "step"})
C1 = P1["config"]
P1_CTX = C1["max_seq_len"]

TOK_PATH = P1_TOKENIZER_OVERRIDE or (find_files("bpe_tokenizer.json") or [None])[0]
assert TOK_PATH, "bpe_tokenizer.json not found in inputs"

class ByteLevelBPETokenizer:
    """Identical to the Phase-1 tokenizer (whitespace words, byte-level BPE, </s> after every word)."""
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            self.special_to_id[tok] = 256 + i
            self.id_to_bytes[256 + i] = tok.encode("utf-8")
    @property
    def vocab_size(self): return len(self.id_to_bytes)
    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    def encode_word(self, word):
        seq = list(word.encode("utf-8"))
        while len(seq) > 1:
            mergeable = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not mergeable: break
            pair = min(mergeable, key=lambda p: self.merges[p])
            seq = self._apply_merge(seq, pair, self.merges[pair])
        seq.append(self.special_to_id["</s>"])
        return seq
    def decode(self, ids):
        eos = self.special_to_id["</s>"]; words, cur = [], []
        flush = lambda: words.append(b"".join(self.id_to_bytes.get(int(t), b"") for t in cur).decode("utf-8", errors="ignore"))
        for i in ids:
            if i == eos:
                if cur: flush(); cur = []
            else: cur.append(i)
        if cur: flush()
        return " ".join(words)
    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f: data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok

tokenizer = ByteLevelBPETokenizer.load(TOK_PATH)
assert tokenizer.vocab_size == C1["vocab_size"], (tokenizer.vocab_size, C1["vocab_size"])
EOS_ID, PAD_ID = tokenizer.special_to_id["</s>"], tokenizer.special_to_id["<pad>"]
BAD_IDS = [tokenizer.special_to_id[t] for t in ("<pad>", "<unk>", "<s>")]

_wcache = {}
def enc_word(w):
    v = _wcache.get(w)
    if v is None:
        v = tuple(tokenizer.encode_word(w))
        if len(_wcache) < 3_000_000: _wcache[w] = v
    return v
def enc_words(words):
    out = []
    for w in words: out.extend(enc_word(w))
    return out
SEP_IDS, END_IDS = enc_words([SEP_WORD]), enc_words([END_WORD])
print("vocab", tokenizer.vocab_size, "| SEP", SEP_IDS, "| END", END_IDS)

# %% [markdown]
# ## 3. Phase-6 data, long-document handling, length statistics

# %%
from huggingface_hub import hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HuggingFace")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

def load_phase6(path, need_summary):
    local = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(local, "r", encoding="utf-8") as f: raw = json.load(f)
    if isinstance(raw, dict):   # {"id": {...}} or {"id": "doc text"}
        raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "document": v} for k, v in raw.items()]
    items = []
    for r in raw:
        doc = next((r[k] for k in ("document", "text", "doc", "judgment") if k in r and r[k]), "")
        item = {"id": str(r.get("id", r.get("doc_id", len(items)))), "document": doc}
        if need_summary: item["summary"] = r.get("summary", r.get("summaries", ""))
        items.append(item)
    return items

train_all = load_phase6(TRAIN_FILE, True)
test_items = load_phase6(TEST_FILE, False)
train_all = [r for r in train_all if r["document"].strip() and r["summary"].strip()]
random.Random(SEED).shuffle(train_all)                 # same split for every method -> comparable val numbers
val_items, train_items = train_all[:N_VAL_DOCS], train_all[N_VAL_DOCS:]
print(f"train {len(train_items)} | val {len(val_items)} | test {len(test_items)}")

_dw = np.array([len(r["document"].split()) for r in train_all]); _sw = np.array([len(r["summary"].split()) for r in train_all])
_pc = lambda a: {p: int(np.percentile(a, p)) for p in (10, 50, 90, 99)}
print("document words  ", _pc(_dw)); print("summary words   ", _pc(_sw))
print(f"~tokens per summary word: {np.mean([len(enc_words(r['summary'].split()[:200]))/max(1,len(r['summary'].split()[:200])) for r in train_all[:100]]):.2f}")
print(f"-> SUM_TOK={SUM_TOK} holds about {int(SUM_TOK/2.7)} summary words; reference median is {int(np.median(_sw))}. Raise SUM_TOK/MAX_CTX if summaries are much longer.")
MAX_SUMMARY_WORDS = int(np.percentile(_sw, 90))
MIN_SUMMARY_WORDS = int(np.percentile(_sw, 10))      # decoding: END marker is blocked before this many words
print(f"decode length window: {MIN_SUMMARY_WORDS}..{MAX_SUMMARY_WORDS} words")

# --------------- budgets ---------------
PROMPT_LEN = CFG["prompt"]["prompt_len"] if METHOD == "prompt" else 0
DOC_TOK = MAX_CTX - PROMPT_LEN - SUM_TOK - len(SEP_IDS)
assert DOC_TOK > 128, "document budget too small: raise MAX_CTX or lower SUM_TOK"
ROPE_LEN = MAX_CTX + 192                                  # buffer length (covers soft prompt + slack)
POS_SCALE = (P1_CTX / MAX_CTX) if (ROPE_MODE == "interp" and MAX_CTX > P1_CTX) else 1.0
print(f"doc budget {DOC_TOK} tok | summary budget {SUM_TOK} tok | rope scale {POS_SCALE:.3f}")

_STOP_LEN = 4
def _salient_words(words, budget):
    """Keep the highest-scoring sentences (tf-based) that fit the token budget, in original order."""
    text = " ".join(words)
    sents = []
    for s in re.split(r"(?<=[.;?!])\s+", text):
        ws = s.split()
        sents.extend(" ".join(ws[i:i + 40]) for i in range(0, len(ws), 40))     # cap sentence length so one run-on can't eat the budget
    sents = [s for s in sents if s]
    tf = Counter(w.lower().strip(".,;:()[]\"'") for w in words if len(w) >= _STOP_LEN)
    n = len(sents); scored = []
    for i, s in enumerate(sents):
        ws = [w.lower().strip(".,;:()[]\"'") for w in s.split() if len(w) >= _STOP_LEN]
        sc = (sum(tf[w] for w in ws) / (len(ws) ** 0.5 + 1e-9)) if ws else 0.0
        if i < 0.1 * n or i > 0.9 * n: sc *= 1.5                 # facts at the start, ruling at the end
        scored.append((sc, i))
    keep, used = set(), 0
    for sc, i in sorted(scored, reverse=True):
        cost = sum(len(enc_word(w)) for w in sents[i].split())
        if used + cost <= budget: keep.add(i); used += cost
    return " ".join(sents[i] for i in sorted(keep)).split()

def select_doc_words(words, budget, mode):
    if mode == "salient":
        if sum(len(enc_word(w)) for w in words[:budget]) <= budget and len(words) <= budget // 2: return words
        return _salient_words(words, budget)
    head_budget = budget if mode == "head" else int(budget * HEAD_FRAC)
    used, h = 0, 0
    while h < len(words) and used + len(enc_word(words[h])) <= head_budget: used += len(enc_word(words[h])); h += 1
    if h == len(words): return words
    if mode == "head": return words[:h]
    tail_budget, t_used, t = budget - used, 0, len(words)
    while t > h and t_used + len(enc_word(words[t - 1])) <= tail_budget: t_used += len(enc_word(words[t - 1])); t -= 1
    return words[:h] + words[t:]

def build_prompt_ids(doc):
    return enc_words(select_doc_words(doc.split(), DOC_TOK, CONTEXT_MODE)) + SEP_IDS

def build_target_ids(summary):
    words, ids = summary.split(), []
    for w in words:
        e = enc_word(w)
        if len(ids) + len(e) + len(END_IDS) > SUM_TOK: break
        ids.extend(e)
    return ids + END_IDS

class SumDataset(Dataset):
    def __init__(self, items):
        self.ex = []
        for r in items:
            p, t = build_prompt_ids(r["document"]), build_target_ids(r["summary"])
            self.ex.append((p + t, [-100] * len(p) + t))
    def __len__(self): return len(self.ex)
    def __getitem__(self, i): return self.ex[i]

def collate(batch):
    L = max(len(x) for x, _ in batch) - 1
    X = torch.full((len(batch), L), PAD_ID, dtype=torch.long); Y = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        X[i, :len(x) - 1] = torch.tensor(x[:-1]); Y[i, :len(y) - 1] = torch.tensor(y[1:])
    return X, Y

t_ = time.time()
train_ds, val_ds = SumDataset(train_items), SumDataset(val_items)
print(f"tokenised in {time.time()-t_:.0f}s | mean example length {np.mean([len(x) for x,_ in train_ds.ex]):.0f} tok | "
      f"{np.mean([sum(1 for v in y if v != -100) for _,y in train_ds.ex]):.0f} target tok")
train_loader = DataLoader(train_ds, batch_size=MICRO_BATCH, shuffle=True, drop_last=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=MICRO_BATCH, shuffle=False, collate_fn=collate)

# %% [markdown]
# ## 4. Decoder model (same parameter names as Phase 1, plus KV-cache / prefix / soft-prompt hooks)

# %%
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0, pos_scale=1.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32) * pos_scale, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None], persistent=False)
    def forward(self, q, k, pos=None):
        if pos is None:
            T = q.size(-2); cos, sin = self.cos[:, :, :T], self.sin[:, :, :T]
        else:                                           # pos: (B, T) absolute positions
            cos, sin = self.cos[0, 0][pos].unsqueeze(1), self.sin[0, 0][pos].unsqueeze(1)
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base, pos_scale):
        super().__init__()
        self.n_heads, self.kv_heads, self.d_head = n_heads, kv_heads, d_model // n_heads
        kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len, rope_base, pos_scale)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, pos=None, key_pad=None, past=None, use_cache=False, prefix=None):
        B, T, C = x.shape
        H, KV, dh = self.n_heads, self.kv_heads, self.d_head
        q = self.q_proj(x).view(B, T, H, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, dh).transpose(1, 2)
        if past is not None: assert pos is not None, "pass absolute positions when using a KV cache"
        q, k = self.rope(q, k, pos)
        q, k = q.to(v.dtype), k.to(v.dtype)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        kv_out = (k, v) if use_cache else None
        S, P = k.size(2), 0
        if prefix is not None:                          # prefix-tuning: learned, un-rotated K/V visible to every query
            pk = prefix[0].to(v.dtype).unsqueeze(0).expand(B, -1, -1, -1)
            pv = prefix[1].to(v.dtype).unsqueeze(0).expand(B, -1, -1, -1)
            k, v, P = torch.cat([pk, k], 2), torch.cat([pv, v], 2), pk.size(2)
        if KV != H:
            k = k.repeat_interleave(H // KV, dim=1); v = v.repeat_interleave(H // KV, dim=1)
        drop = self.dropout if self.training else 0.0
        if P > 0 or key_pad is not None:
            q_pos = torch.arange(S - T, S, device=x.device); k_pos = torch.arange(S, device=x.device)
            m = k_pos[None, :] <= q_pos[:, None]
            if key_pad is not None:
                m = (m[None] & key_pad[:, None, :]) | (k_pos[None, None, :] == q_pos[None, :, None])  # a token always sees itself (no NaN rows)
            else:
                m = m[None].expand(B, -1, -1)
            if P > 0: m = torch.cat([torch.ones(B, T, P, dtype=torch.bool, device=x.device), m], 2)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=m[:, None], dropout_p=drop)
        elif T == S:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            assert T == 1
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=drop)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))), kv_out

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False); self.dropout = nn.Dropout(dropout)
    def forward(self, x): return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base, pos_scale):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base, pos_scale)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x, pos=None, key_pad=None, past=None, use_cache=False, prefix=None):
        a, kv = self.attn(self.norm_attn(x), pos, key_pad, past, use_cache, prefix)
        x = x + a
        return x + self.ffn(self.norm_ffn(x)), kv

class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base, pos_scale):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base, pos_scale)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.soft_prompt = None        # set by prompt-tuning
        self.prefix_encoder = None     # set by prefix-tuning

    def _backbone(self, x, pos=None, key_pad=None, past=None, use_cache=False):
        assert x.size(1) + (0 if past is None else past[0][0].size(2)) <= self.max_seq_len, "sequence longer than RoPE buffer"
        prefix = self.prefix_encoder() if self.prefix_encoder is not None else None
        new_past = []
        for i, blk in enumerate(self.blocks):
            x, kv = blk(x, pos, key_pad, None if past is None else past[i], use_cache, None if prefix is None else prefix[i])
            new_past.append(kv)
        return self.final_norm(x), (new_past if use_cache else None)

    def _embed(self, idx):
        x = self.token_emb(idx)
        if self.soft_prompt is not None:
            x = torch.cat([self.soft_prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], 1)
        return x

    def forward(self, idx, targets=None):
        """targets=None -> full logits. With targets (-100 = ignore): loss + logits ONLY at supervised positions."""
        P = 0 if self.soft_prompt is None else self.soft_prompt.size(0)
        h, _ = self._backbone(self._embed(idx))
        h = h[:, P:]
        if targets is None: return self.lm_head(h), None, None
        sel = targets != -100
        logits = self.lm_head(h[sel])
        return logits, F.cross_entropy(logits.float(), targets[sel]), sel

    @torch.no_grad()
    def prefill(self, idx, valid):
        """idx/valid: (B, L) left-padded prompt. Returns last-position logits + cache state."""
        x = self._embed(idx)
        if self.soft_prompt is not None:
            valid = torch.cat([torch.ones(idx.size(0), self.soft_prompt.size(0), dtype=torch.bool, device=idx.device), valid], 1)
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        h, past = self._backbone(x, pos, valid, None, True)
        return self.lm_head(h[:, -1]), past, valid, valid.sum(1)

    @torch.no_grad()
    def step(self, tok, past, key_pad, next_pos):
        key_pad = torch.cat([key_pad, torch.ones(tok.size(0), 1, dtype=torch.bool, device=tok.device)], 1)
        h, past = self._backbone(self.token_emb(tok[:, None]), next_pos[:, None], key_pad, past, True)
        return self.lm_head(h[:, -1]), past, key_pad, next_pos + 1

def build_base(n_layers=None):
    """Fresh copy of the Phase-1 decoder (weights loaded), long-context RoPE buffers."""
    m = DecoderLM(C1["vocab_size"], C1["d_model"], C1["n_layers"], C1["n_heads"], C1["kv_heads"], C1["d_ff"],
                  ROPE_LEN, C1["dropout"], C1["rope_base"], POS_SCALE)
    m.load_state_dict(P1["state_dict"], strict=True)        # RoPE buffers are non-persistent -> keys match Phase 1
    return m

def n_params(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only))

# %% [markdown]
# ## 5. The adaptation methods

# %%
def freeze(model):
    for p in model.parameters(): p.requires_grad = False

# ---------- LoRA : W' = W0 + (alpha/r) B A ----------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale
    @torch.no_grad()
    def merged_weight(self): return self.base.weight + self.scale * (self.B @ self.A)

# ---------- LoRMA : W' = I_pi(B A) W0   (multiplicative, permutation rank inflation, Bihany et al. 2025) ----------
class LoRMALinear(nn.Module):
    """Update matrix M = I_pi(BA) is (out x out): row i of BA is cyclically rotated by i, which lifts the rank of the
    rank-r product BA to (near) full rank. Init follows the paper: B[:,0]=1, A[0,0]=1, rest of A = 0 => I_pi(BA) = I."""
    def __init__(self, base, r):
        super().__init__()
        self.base, d = base, base.out_features
        B = torch.randn(d, r) * 0.02; B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        idx = (torch.arange(d)[None, :] - torch.arange(d)[:, None]) % d       # M[i, j] = BA[i, (j - i) mod d]
        self.register_buffer("idx", idx, persistent=False)
    def update_matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, x): return F.linear(x, self.update_matrix() @ self.base.weight)   # W0 here has no bias
    @torch.no_grad()
    def merged_weight(self): return self.update_matrix() @ self.base.weight

def _swap_modules(model, predicate, make):
    for pname, parent in list(model.named_modules()):
        for cname, child in list(parent.named_children()):
            if predicate(child): setattr(parent, cname, make(child))

def apply_lora_like(model, cfg, kind):
    count = 0
    for blk in model.blocks:
        for mod in (blk.attn, blk.ffn):
            for name in cfg["targets"]:
                if hasattr(mod, name) and isinstance(getattr(mod, name), nn.Linear):
                    lin = getattr(mod, name)
                    setattr(mod, name, LoRMALinear(lin, cfg["rank"]) if kind == "lorma"
                            else LoRALinear(lin, cfg["rank"], cfg["alpha"], cfg["dropout"]))
                    count += 1
    return count

@torch.no_grad()
def merge_adapters(model):
    """Fold LoRA/LoRMA into the dense weights -> no extra inference cost. (Irreversible; call after training.)"""
    def make(m):
        m.base.weight.data = m.merged_weight().to(m.base.weight.dtype); return m.base
    _swap_modules(model, lambda c: isinstance(c, (LoRALinear, LoRMALinear)), make)

# ---------- Houlsby adapters ----------
class Adapter(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.down, self.up = nn.Linear(d, m), nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)         # starts as identity
    def forward(self, h): return h + self.up(F.gelu(self.down(h)))

class AdapterWrap(nn.Module):
    def __init__(self, inner, adapter, returns_tuple):
        super().__init__(); self.inner, self.adapter, self.rt = inner, adapter, returns_tuple
    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        return (self.adapter(out[0]), out[1]) if self.rt else self.adapter(out)

def apply_houlsby(model, cfg):
    d = C1["d_model"]
    for blk in model.blocks:      # adapter after attention output and after FFN output, both before the residual add
        blk.attn = AdapterWrap(blk.attn, Adapter(d, cfg["bottleneck"]), True)
        blk.ffn = AdapterWrap(blk.ffn, Adapter(d, cfg["bottleneck"]), False)
    if cfg["train_norms"]:
        for n, p in model.named_parameters():
            if ".norm_attn." in n or ".norm_ffn." in n or n.startswith("final_norm."): p.requires_grad = True

# ---------- Prefix-tuning ----------
class PrefixEncoder(nn.Module):
    """Embedding(P, h) -> MLP -> per-layer (K, V) prefixes (Li & Liang's re-parametrisation)."""
    def __init__(self, n_layers, kv_heads, d_head, plen, hidden):
        super().__init__()
        self.L, self.KV, self.dh, self.P = n_layers, kv_heads, d_head, plen
        self.emb = nn.Embedding(plen, hidden)
        self.mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, n_layers * 2 * kv_heads * d_head))
        nn.init.normal_(self.mlp[2].weight, std=0.02); nn.init.zeros_(self.mlp[2].bias)
    def forward(self):
        o = self.mlp(self.emb.weight).view(self.P, self.L, 2, self.KV, self.dh).permute(1, 2, 3, 0, 4)   # (L,2,KV,P,dh)
        return [(o[l, 0], o[l, 1]) for l in range(self.L)]

def apply_prefix(model, cfg):
    model.prefix_encoder = PrefixEncoder(len(model.blocks), C1["kv_heads"], C1["d_model"] // C1["n_heads"], cfg["prefix_len"], cfg["hidden"])

# ---------- Prompt-tuning ----------
def apply_prompt(model, cfg):
    g = torch.Generator().manual_seed(SEED)
    ids = torch.randint(260, min(model.token_emb.num_embeddings, 3000), (cfg["prompt_len"],), generator=g)   # frequent BPE tokens
    model.soft_prompt = nn.Parameter(model.token_emb.weight[ids].detach().clone())

# ---------- builders ----------
def build_method_model(method):
    m = build_base()
    cfg = CFG[method]
    if method == "kd":        # student: alternate Phase-1 layers (6 of 12), everything trainable
        m.blocks = nn.ModuleList([m.blocks[i] for i in cfg["student_layers"]])
        return m.to(DEVICE)
    freeze(m)
    if method in ("lorma", "lora"): print("wrapped linears:", apply_lora_like(m, cfg, method))
    elif method == "houlsby": apply_houlsby(m, cfg)
    elif method == "prefix": apply_prefix(m, cfg)
    elif method == "prompt": apply_prompt(m, cfg)
    return m.to(DEVICE)

def trainable_state(m): return {n: p.detach().cpu().clone() for n, p in m.named_parameters() if p.requires_grad}
def load_trainable(m, state):
    own = dict(m.named_parameters()); missing = [k for k in state if k not in own]
    assert not missing, f"checkpoint keys not in model: {missing[:5]}"
    with torch.no_grad():
        for k, v in state.items(): own[k].copy_(v)

# %% [markdown]
# ## 6. Build the model for this run (+ teacher for KD)

# %%
model = build_method_model(METHOD)
total_p, train_p = n_params(model), n_params(model, True)
print(f"{METHOD}: total {total_p/1e6:.2f}M params | trainable {train_p/1e6:.3f}M ({100*train_p/total_p:.2f}%)")

teacher = None
if METHOD == "kd":
    assert TEACHER != "prompt" and TEACHER != "kd" and TEACHER in CFG
    tpaths = [p for p in find_files("best_trainable.pt", [INPUT_ROOT, OUT_ROOT]) if os.sep + TEACHER + os.sep in p]
    assert tpaths, f"teacher '{TEACHER}' not found: finish that run (or attach its output) first"
    teacher = build_method_model(TEACHER)
    tck = torch.load(max(tpaths, key=os.path.getmtime), map_location="cpu", weights_only=False)
    load_trainable(teacher, tck["state"])
    if TEACHER in ("lorma", "lora"): merge_adapters(teacher)
    teacher.eval(); freeze(teacher)
    print(f"teacher={TEACHER} ({n_params(teacher)/1e6:.1f}M) -> student ({total_p/1e6:.1f}M), size ratio {total_p/n_params(teacher):.2f}")

# %% [markdown]
# ## 7. Training (best-validation checkpoint is kept)

# %%
def kd_loss(s_logits, t_logits, targets_sel, T, alpha):
    ce = F.cross_entropy(s_logits.float(), targets_sel)
    kl = F.kl_div(F.log_softmax(s_logits.float() / T, -1), F.softmax(t_logits.float() / T, -1), reduction="batchmean") * (T * T)
    return alpha * ce + (1 - alpha) * kl

def step_loss(x, y):
    with torch.autocast(device_type="cuda", enabled=USE_CUDA):
        logits, loss, sel = model(x, y)
        if teacher is not None:
            with torch.no_grad(): t_logits, _, _ = teacher(x, y)
            loss = kd_loss(logits, t_logits, y[sel], CFG["kd"]["temperature"], CFG["kd"]["alpha"])
    return loss

@torch.no_grad()
def eval_loss(max_batches=40):
    model.eval(); tot, n = 0.0, 0
    for i, (x, y) in enumerate(val_loader):
        if i >= max_batches: break
        with torch.autocast(device_type="cuda", enabled=USE_CUDA):
            _, loss, _ = model(x.to(DEVICE), y.to(DEVICE))          # plain CE on gold summary (also for KD)
        tot += loss.item(); n += 1
    model.train(); return tot / max(1, n)

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=CFG[METHOD]["lr"], weight_decay=0.0 if METHOD != "kd" else 0.01, betas=(0.9, 0.98))
scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)
steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
TOTAL_STEPS = steps_per_epoch * EPOCHS
WARMUP = max(1, int(WARMUP_RATIO * TOTAL_STEPS))
def lr_at(s):
    if s < WARMUP: return CFG[METHOD]["lr"] * (s + 1) / WARMUP
    return 0.5 * CFG[METHOD]["lr"] * (1 + math.cos(math.pi * (s - WARMUP) / max(1, TOTAL_STEPS - WARMUP)))

print(f"{len(train_loader)} micro-batches/epoch -> {TOTAL_STEPS} optimiser steps ({EPOCHS} epochs)")
step, best_val, best_state, hist = 0, float("inf"), None, {"step": [], "train": [], "val_step": [], "val": []}
if USE_CUDA: torch.cuda.reset_peak_memory_stats()
t0 = time.time(); tok_seen = 0; stopped_early = False
model.train(); data_iter = iter(train_loader)
while step < TOTAL_STEPS:
    opt.zero_grad(set_to_none=True); acc = 0.0
    for _ in range(GRAD_ACCUM):
        try: x, y = next(data_iter)
        except StopIteration: data_iter = iter(train_loader); x, y = next(data_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)
        loss = step_loss(x, y) / GRAD_ACCUM
        scaler.scale(loss).backward(); acc += loss.item(); tok_seen += x.numel()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
    for g in opt.param_groups: g["lr"] = lr_at(step)
    scaler.step(opt); scaler.update(); step += 1
    hist["step"].append(step); hist["train"].append(acc)
    if step % 10 == 0:
        print(f"step {step}/{TOTAL_STEPS} loss {acc:.4f} lr {lr_at(step):.2e} | {tok_seen/(time.time()-t0):.0f} tok/s")
    if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
        v = eval_loss(); hist["val_step"].append(step); hist["val"].append(v)
        print(f"  eval step {step}: val CE {v:.4f} (ppl {math.exp(v):.2f})")
        if v < best_val:
            best_val, best_state = v, trainable_state(model)
            torch.save({"method": METHOD, "cfg": CFG[METHOD], "state": best_state, "step": step, "val_loss": v},
                       os.path.join(OUT_DIR, "best_trainable.pt"))
    if time.time() - t0 > TIME_BUDGET_H * 3600 and step < TOTAL_STEPS:
        print("!! time budget hit, stopping early"); stopped_early = True; break
train_hours = (time.time() - t0) / 3600
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9 if USE_CUDA else 0.0
if best_state is None: best_state = trainable_state(model); best_val = eval_loss()
load_trainable(model, best_state)                       # checkpoint selection: best validation loss, not last step
print(f"done: {train_hours:.2f} h | best val CE {best_val:.4f} | peak GPU mem {peak_mem_gb:.1f} GB")

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4)); plt.plot(hist["step"], hist["train"], label="train", alpha=.6)
plt.plot(hist["val_step"], hist["val"], "o-", label="val"); plt.legend(); plt.grid(True); plt.title(METHOD)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "loss.png"), dpi=150); plt.close()

# %% [markdown]
# ## 8. Generation (KV-cached, batched, left-padded)

# %%
if METHOD in ("lorma", "lora"): merge_adapters(model)    # best weights are already saved; merged model decodes at dense speed
model.eval()

@torch.no_grad()
def generate(token_lists, temperature=0.0, top_k=0, top_p=1.0, rep_pen=1.0, rep_window=256, max_new=SUM_TOK):
    B, L = len(token_lists), max(len(t) for t in token_lists)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long); valid = torch.zeros(B, L, dtype=torch.bool)
    for i, t in enumerate(token_lists): ids[i, L - len(t):] = torch.tensor(t); valid[i, L - len(t):] = True
    ids, valid = ids.to(DEVICE), valid.to(DEVICE)
    end = torch.tensor(END_IDS, device=DEVICE); ne = len(END_IDS)
    gen = torch.empty((B, 0), dtype=torch.long, device=DEVICE); done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    with torch.autocast(device_type="cuda", enabled=USE_CUDA):
        logits, past, key_pad, next_pos = model.prefill(ids, valid)
    for _ in range(max_new):
        logits = logits.float()
        if rep_pen != 1.0 and gen.size(1) > 0:
            eos_l = logits[:, EOS_ID].clone(); recent = gen[:, -rep_window:]
            sc = torch.gather(logits, 1, recent); sc = torch.where(sc > 0, sc / rep_pen, sc * rep_pen)
            logits.scatter_(1, recent, sc); logits[:, EOS_ID] = eos_l
        logits[:, BAD_IDS] = float("-inf")
        prev = gen[:, -1] if gen.size(1) > 0 else ids[:, -1]
        logits[prev == EOS_ID, EOS_ID] = float("-inf")                 # no empty words
        short = (prev == EOS_ID) & ((gen == EOS_ID).sum(1) < MIN_SUMMARY_WORDS)
        logits[short, END_IDS[0]] = float("-inf")                      # no premature END marker
        if temperature <= 0: nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            sp, si = torch.sort(torch.softmax(logits, -1), descending=True, dim=-1)
            sp = sp.masked_fill((torch.cumsum(sp, -1) - sp) > top_p, 0.0); sp = sp / sp.sum(-1, keepdim=True)
            nxt = torch.gather(si, -1, torch.multinomial(sp, 1))
        nxt[done] = PAD_ID
        gen = torch.cat([gen, nxt], 1)
        if gen.size(1) >= ne: done |= (gen[:, -ne:] == end).all(1)
        if bool(done.all()): break
        with torch.autocast(device_type="cuda", enabled=USE_CUDA):
            logits, past, key_pad, next_pos = model.step(nxt[:, 0], past, key_pad, next_pos)
    outs = []
    for row in gen.tolist():
        words = tokenizer.decode(row).split()
        if END_WORD in words: words = words[:words.index(END_WORD)]
        outs.append(" ".join(words[:MAX_SUMMARY_WORDS]))
    return outs

def summarise(items, cfg, desc=""):
    prompts = {r["id"]: build_prompt_ids(r["document"]) for r in items}
    order = sorted(prompts, key=lambda k: len(prompts[k])); res, t_ = {}, time.time()
    for s in range(0, len(order), GEN_BATCH):
        chunk = order[s:s + GEN_BATCH]
        for k, txt in zip(chunk, generate([prompts[k] for k in chunk], **cfg)): res[k] = txt
        if (s // GEN_BATCH) % 10 == 0: print(f"  {desc} {min(s+GEN_BATCH, len(order))}/{len(order)} ({time.time()-t_:.0f}s)")
    for r in items:                                          # never submit an empty summary
        if not res[r["id"]].strip(): res[r["id"]] = " ".join(r["document"].split()[:60])
    return res

# %% [markdown]
# ## 9. Local evaluation (ROUGE-1/L + BERTScore) and decoding-grid selection

# %%
def compute_metrics(preds, refs, do_bert=True):
    out = {}
    try:
        from rouge_score import rouge_scorer
        sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"])
        r = [sc.score(ref, p) for p, ref in zip(preds, refs)]
        out["rouge1_f"] = float(np.mean([x["rouge1"].fmeasure for x in r])); out["rougeL_f"] = float(np.mean([x["rougeL"].fmeasure for x in r]))
    except Exception as e: print("rouge unavailable:", e)
    if do_bert:
        try:
            from bert_score import score as bs
            _, _, f1 = bs(preds, refs, lang="en", batch_size=16, device=str(DEVICE), verbose=False)
            out["bertscore_f1"] = float(f1.mean())
        except Exception as e: print("bert_score unavailable:", e)
    return out

val_eval = val_items[:N_VAL_GEN]; val_refs = [r["summary"] for r in val_eval]
grid_res = []
for cfg in DECODE_GRID:
    torch.manual_seed(SEED); t_ = time.time()
    preds = summarise(val_eval, cfg, "val"); m = compute_metrics([preds[r["id"]] for r in val_eval], val_refs)
    m["gen_seconds"] = time.time() - t_; grid_res.append((m, cfg)); print(cfg, "->", m)
key = lambda mc: mc[0].get("bertscore_f1", mc[0].get("rougeL_f", 0.0))
BEST_METRICS, BEST_DEC = max(grid_res, key=key)
print("BEST decoding:", BEST_DEC, BEST_METRICS)
best_preds = summarise(val_eval, BEST_DEC, "val-best")
json.dump({r["id"]: {"pred": best_preds[r["id"]], "ref": r["summary"]} for r in val_eval},
          open(os.path.join(OUT_DIR, "val_predictions.json"), "w"), ensure_ascii=False, indent=1)
print("avg pred words", np.mean([len(best_preds[r['id']].split()) for r in val_eval]), "| avg ref words", np.mean([len(x.split()) for x in val_refs]))

# %% [markdown]
# ## 10. Test predictions -> `submission.zip`  (list format, `prediction.json` at the zip root)

# %%
torch.manual_seed(SEED); t_ = time.time()
test_preds = summarise(test_items, BEST_DEC, "test"); infer_s = time.time() - t_
submission = [{"id": r["id"], "summary": test_preds[r["id"]]} for r in test_items]
assert len(submission) == len(test_items) and len({s["id"] for s in submission}) == len(test_items)
assert all(s["summary"].strip() for s in submission)
PRED_PATH, ZIP_PATH = os.path.join(OUT_DIR, "prediction.json"), os.path.join(OUT_DIR, "submission.zip")
json.dump(submission, open(PRED_PATH, "w"), ensure_ascii=False, indent=1)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf: zf.write(PRED_PATH, arcname="prediction.json")
print("SUBMIT:", ZIP_PATH, "| docs:", len(submission), "| avg words:", np.mean([len(s["summary"].split()) for s in submission]))

# %% [markdown]
# ## 11. Log this run (for the report's efficiency / comparison tables)

# %%
row = {"method": METHOD, "teacher": TEACHER if METHOD == "kd" else "", "total_params_M": round(total_p / 1e6, 2),
       "trainable_params_M": round(train_p / 1e6, 3), "trainable_pct": round(100 * train_p / total_p, 3),
       "best_val_ce": round(best_val, 4), "val_rouge1_f": round(BEST_METRICS.get("rouge1_f", float("nan")), 4),
       "val_rougeL_f": round(BEST_METRICS.get("rougeL_f", float("nan")), 4), "val_bertscore_f1": round(BEST_METRICS.get("bertscore_f1", float("nan")), 4),
       "train_hours": round(train_hours, 3), "peak_gpu_gb": round(peak_mem_gb, 2), "test_infer_seconds": round(infer_s, 1),
       "max_ctx": MAX_CTX, "sum_tok": SUM_TOK, "context_mode": CONTEXT_MODE, "rope_mode": ROPE_MODE, "epochs": EPOCHS,
       "decoding": json.dumps(BEST_DEC), "stopped_early": stopped_early}
new = not os.path.exists(RESULTS_CSV)
with open(RESULTS_CSV, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row)); (new and w.writeheader()); w.writerow(row)
print(row)


Writing /kaggle/working/phase6_run.py


In [2]:
import os, sys, time, glob, shutil, threading, subprocess, csv

WORK = os.environ.get("P6_WORK", "/kaggle/working")
SCRIPT = os.path.join(WORK, "phase6_run.py")
OUT_ROOT = os.path.join(WORK, "cs781_phase6")
SUBS = os.path.join(WORK, "submissions")
os.makedirs(OUT_ROOT, exist_ok=True); os.makedirs(SUBS, exist_ok=True)

# ---------------- settings ----------------
QUEUES = {                      # one queue per GPU; queues run in parallel, methods inside a queue run one after another
    0: ["lorma", "lora", "kd"],          # kd must come after lorma (its teacher) -> same queue
    1: ["houlsby", "prefix", "prompt"],
}
EPOCHS = 3                      # lower to 2 if you are short on time
TRAIN_HOURS_PER_METHOD = 1.0    # hard cap on the TRAINING loop of each method (it still decodes + writes a submission after)
# If your session shows only ONE GPU, use: QUEUES = {0: ["lorma","lora","houlsby","prefix","prompt","kd"]}  and TRAIN_HOURS_PER_METHOD = 0.6
# -------------------------------------------

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HuggingFace")
except Exception as e:
    print("no kaggle secret:", e)

status = {}
def run_queue(gpu, methods):
    for m in methods:
        zip_path = os.path.join(OUT_ROOT, m, "submission.zip")
        if os.path.exists(zip_path):
            status[m] = "skipped (already done)"; continue
        env = dict(os.environ, P6_METHOD=m, P6_EPOCHS=str(EPOCHS), P6_TIME_H=str(TRAIN_HOURS_PER_METHOD),
                   CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", CS781_OUT=OUT_ROOT)
        t0 = time.time(); status[m] = f"running on GPU {gpu}"
        with open(os.path.join(WORK, f"log_{m}.txt"), "w") as log:
            rc = subprocess.run([sys.executable, SCRIPT], env=env, stdout=log, stderr=subprocess.STDOUT).returncode
        ok = rc == 0 and os.path.exists(zip_path)
        status[m] = f"{'OK' if ok else 'FAILED (see log_' + m + '.txt)'} in {(time.time()-t0)/60:.0f} min"
        if ok: shutil.copy(zip_path, os.path.join(SUBS, f"{m}_submission.zip"))

threads = [threading.Thread(target=run_queue, args=(g, ms)) for g, ms in QUEUES.items()]
for t in threads: t.start()
while any(t.is_alive() for t in threads):
    time.sleep(180)
    print(time.strftime("%H:%M"), {m: s for m, s in status.items()})
    for m, s in status.items():
        if s.startswith("running"):
            try: print(f"   [{m}]", open(os.path.join(WORK, f"log_{m}.txt")).read().strip().splitlines()[-1][:160])
            except Exception: pass
for t in threads: t.join()

print("\n===== SUMMARY =====")
for m, s in status.items(): print(f"{m:8s} {s}")
print("\nsubmission zips to upload to Codabench (one per method):")
for z in sorted(glob.glob(os.path.join(SUBS, "*.zip"))): print("  ", z)
csv_path = os.path.join(OUT_ROOT, "phase6_results.csv")
if os.path.exists(csv_path):
    print("\n" + open(csv_path).read())


08:53 {'lorma': 'running on GPU 0', 'houlsby': 'running on GPU 1'}
   [lorma] step 120/1287 loss 1.6827 lr 1.99e-04 | 15826 tok/s
   [houlsby]   eval step 100: val CE 1.7537 (ppl 5.78)
08:56 {'lorma': 'running on GPU 0', 'houlsby': 'running on GPU 1'}
   [lorma] step 290/1287 loss 1.7266 lr 1.84e-04 | 15905 tok/s
   [houlsby] step 260/1287 loss 1.7294 lr 4.69e-04 | 14289 tok/s
08:59 {'lorma': 'running on GPU 0', 'houlsby': 'running on GPU 1'}
   [lorma] step 460/1287 loss 1.6158 lr 1.53e-04 | 15790 tok/s
   [houlsby] step 410/1287 loss 1.5320 lr 4.08e-04 | 14156 tok/s
09:02 {'lorma': 'running on GPU 0', 'houlsby': 'running on GPU 1'}
   [lorma] step 630/1287 loss 1.7366 lr 1.12e-04 | 15740 tok/s
   [houlsby] step 570/1287 loss 1.6339 lr 3.17e-04 | 14203 tok/s
09:05 {'lorma': 'running on GPU 0', 'houlsby': 'running on GPU 1'}
   [lorma] step 810/1287 loss 1.6953 lr 6.61e-05 | 15718 tok/s
   [houlsby] step 730/1287 loss 1.6769 lr 2.15e-04 | 14155 tok/s
09:08 {'lorma': 'running on GPU 0',